In [1]:
import os
import shutil
import sys

# Tell PySpark where the Windows translator files live
os.environ['HADOOP_HOME'] = 'C:\\hadoop'

# Make PySpark use the same Python executable as this notebook kernel
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(f"Notebook Python: {sys.executable}")

# Check Java before importing/starting Spark, so the notebook fails clearly instead of hanging
if not os.environ.get("JAVA_HOME") and shutil.which("java") is None:
    raise RuntimeError("PySpark needs Java before SparkSession can start. Install a JDK and set JAVA_HOME, then restart VS Code/Jupyter.")

print("PySpark setup check passed.")

Notebook Python: c:\Users\zraym\AppData\Local\Programs\Python\Python314\python.exe
PySpark setup check passed.


In [2]:
print("About to import SparkSession...", flush=True)
from pyspark.sql import SparkSession
print("SparkSession imported.", flush=True)

from pyspark.sql.functions import monotonically_increasing_id, col, when
print("PySpark SQL functions imported.", flush=True)

# start up the spark engine
print("About to start Spark engine...", flush=True)
spark = SparkSession.builder \
    .appName("SECOM_Data_Engine") \
    .master("local[1]") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()

print("Spark Engine Online!", flush=True)

# raw files ingestion
# inferSchema = True to automatically figure out if a column is a float, int, or string
sensors_df = spark.read.csv('../data/raw/secom.data', sep=' ', header=False, inferSchema=True)
labels_df = spark.read.csv('../data/raw/secom_labels.data', sep=' ', header=False, inferSchema=True)

# rename the label colums because pysparks 
# defaults to _c0, _c1, etc
labels_df = labels_df.withColumnRenamed("_c0", "Result").withColumnRenamed("_c1", "Timestamp")

# changing all the -1 pass to 0 just like before
labels_df = labels_df.withColumn("Result", when(col("Result") == -1, 0).otherwise(col("Result")))

# assigning a id number to every row, saved in row_id for both sensors and labels
sensors_df = sensors_df.withColumn("row_id", monotonically_increasing_id())
labels_df = labels_df.withColumn("row_id", monotonically_increasing_id())

# matching each id with the other id, sipping them together
# and then deleting the row id as it is not useful anymore
df_spark = labels_df.join(sensors_df, on="row_id", how="inner").drop("row_id")

# pyspark didnt actually do anything yet it jsut writes down instructions to do it. 
row_count = df_spark.count() # action command, execute all the stitching instructions
col_count = len(df_spark.columns)
print(f"\nDistributed Dataset Shape: ({row_count}, {col_count})")

# look at the data
df_spark.select("Result", "Timestamp", "_c0", "_c1", "_c2").show(5)

About to import SparkSession...
SparkSession imported.
PySpark SQL functions imported.
About to start Spark engine...
Spark Engine Online!

Distributed Dataset Shape: (1567, 592)
+------+-------------------+-------+-------+---------+
|Result|          Timestamp|    _c0|    _c1|      _c2|
+------+-------------------+-------+-------+---------+
|     0|19/07/2008 11:55:00|3030.93| 2564.0|2187.7333|
|     0|19/07/2008 12:32:00|3095.78|2465.14|2230.4222|
|     1|19/07/2008 13:17:00|2932.61|2559.94|2186.4111|
|     0|19/07/2008 14:43:00|2988.72| 2479.9|2199.0333|
|     0|19/07/2008 15:22:00|3032.24|2502.87|2233.3667|
+------+-------------------+-------+-------+---------+
only showing top 5 rows


In [3]:
from pyspark.sql.functions import col, count, when, isnan, isnull, min as spark_min, max as spark_max

print("Auditing distributed cluster data (this takes some heavy lifting)...")

# isolate the sensor colummns ignore labels
sensor_cols = [c for c in df_spark.columns if c not in ["Result", "Timestamp"]]
total_rows = df_spark.count()

# Audit columns in small batches so Spark does not build one huge aggregation plan.
# A sensor is constant if its non-missing min and max are the same.
audit_results = {}
batch_size = 50

for start in range(0, len(sensor_cols), batch_size):
    batch_cols = sensor_cols[start:start + batch_size]
    exprs = []

    for c in batch_cols:
        missing_value = isnull(col(c)) | isnan(col(c))
        valid_value = when(~missing_value, col(c))

        exprs.append(count(when(missing_value, c)).alias(f"{c}_nulls"))
        exprs.append(count(valid_value).alias(f"{c}_valids"))
        exprs.append(spark_min(valid_value).alias(f"{c}_min"))
        exprs.append(spark_max(valid_value).alias(f"{c}_max"))

    audit_results.update(df_spark.agg(*exprs).first().asDict())

# looping through every sensor name, calculate the percentage of missing data
# missing more than 50%, dead sensor variance
cols_to_drop = []
for c in sensor_cols:
    null_pct = audit_results[f"{c}_nulls"] / total_rows
    valid_count = audit_results[f"{c}_valids"]
    min_value = audit_results[f"{c}_min"]
    max_value = audit_results[f"{c}_max"]
    
    # If missing > 50% OR if it is a dead sensor with no variation
    if null_pct > 0.50 or valid_count <= 1 or min_value == max_value:
        cols_to_drop.append(c) # hit list

print(f"Found {len(cols_to_drop)} dead or heavily missing sensor columns.")

# take the hit list an remove the columns from dataset
df_spark_clean = df_spark.drop(*cols_to_drop) # asterisk helps with unpacking the labels

# Verify the new shape
clean_row_count = df_spark_clean.count()
clean_col_count = len(df_spark_clean.columns)
print(f"Cleaned Distributed Shape: ({clean_row_count}, {clean_col_count})")

Auditing distributed cluster data (this takes some heavy lifting)...
Found 144 dead or heavily missing sensor columns.
Cleaned Distributed Shape: (1567, 448)


In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load the notebook-local .env file first, then fall back to dotenv's normal search.
env_path = Path('.env')
if env_path.exists():
    load_dotenv(env_path, override=True)
else:
    load_dotenv(override=True)

# Now you can safely pull them into your code
URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USER")
PASSWORD = os.getenv("NEO4J_PASSWORD")

print("Credentials loaded securely!")
print(f"Neo4j URI host: {URI.split('://', 1)[-1].split(':', 1)[0] if URI else 'missing'}")

Credentials loaded securely!
Neo4j URI host: 754a8619.databases.neo4j.io


In [11]:
import os
import socket
import joblib
from urllib.parse import urlparse

try:
    from neo4j import GraphDatabase
except ModuleNotFoundError as error:
    raise RuntimeError("Install the Neo4j driver first with: %pip install neo4j") from error

# Load intelligence layer
deployed_asset = joblib.load('secom_xgboost_production_v1.pkl')
ml_pipeline = deployed_asset['pipeline']
threshold = deployed_asset['optimal_threshold']

# Grab a 10-row sample of the clean distributed data and bring it into memory
sample_data = df_spark_clean.limit(10).toPandas()

# Spark names sensor columns like _c0, while the sklearn pipeline was trained on string names like 0.
# Rename and order the inference columns to match the fitted pipeline exactly.
inference_features = sample_data.drop(columns=['Timestamp', 'Result']).rename(
    columns=lambda name: name[2:] if isinstance(name, str) and name.startswith('_c') else str(name)
)
expected_features = list(getattr(ml_pipeline, 'feature_names_in_', []))
if expected_features:
    missing_features = [feature for feature in expected_features if feature not in inference_features.columns]
    if missing_features:
        raise ValueError(f"Missing model features after Spark cleanup: {missing_features[:10]}")
    inference_features = inference_features[expected_features]

# Generate the ML Risk Scores
risk_scores = ml_pipeline.predict_proba(inference_features)[:, 1]
sample_data = sample_data.assign(ML_Risk_Score=risk_scores)

print("Intelligence layer applied. Booting up Neo4j connection...")

# connect to Neo4j using environment variables instead of hard-coded credentials
URI = os.environ.get('NEO4J_URI')
USER = os.environ.get('NEO4J_USER', 'neo4j')
PASSWORD = os.environ.get('NEO4J_PASSWORD')

neo4j_enabled = bool(URI and PASSWORD)

if not neo4j_enabled:
    print(
        "Neo4j credentials not set. ML risk scores were generated, "
        "but graph upload will be skipped."
    )
    print("To enable upload, set NEO4J_URI and NEO4J_PASSWORD, then rerun this cell.")
else:
    parsed_uri = urlparse(URI)
    if parsed_uri.scheme not in {'neo4j+s', 'neo4j+ssc', 'bolt+s', 'bolt+ssc'}:
        raise ValueError(f"Unexpected Neo4j URI scheme: {parsed_uri.scheme}. Aura usually uses neo4j+s://...")
    if not parsed_uri.hostname:
        raise ValueError("NEO4J_URI is missing a hostname. It should look like neo4j+s://<id>.databases.neo4j.io")
    try:
        socket.getaddrinfo(parsed_uri.hostname, parsed_uri.port or 7687)
    except socket.gaierror as error:
        raise RuntimeError(
            f"Could not DNS-resolve Neo4j host '{parsed_uri.hostname}'. "
            "This usually means the Aura URI is wrong, the database instance was deleted/renamed, "
            "or your DNS/network is blocking *.databases.neo4j.io. Copy the URI again from Neo4j Aura."
        ) from error
    AUTH = (USER, PASSWORD)
    driver = GraphDatabase.driver(URI, auth=AUTH)
    driver.verify_connectivity()
    print("Neo4j connection verified.")

# ==========================================
# 3. DEFINE THE GRAPH STRUCTURE (CYPHER)
# ==========================================
def build_digital_twin(tx, wafer_id, timestamp, actual_result, risk_score, sensor_dict):
    query = """
    // 1. Create the physical Process Tool
    MERGE (m:Machine {tool_id: 'Etch_Chamber_A'})
    
    // 2. Create the Wafer Lot (Now with the ML Risk Score attached!)
    CREATE (w:Wafer {
        wafer_id: $wafer_id, 
        timestamp: $timestamp, 
        actual_yield: $actual_result,
        ml_risk_score: $risk_score
    })
    
    // 3. Link the Machine to the Wafer
    CREATE (m)-[:PROCESSED]->(w)
    
    // 4. Create the Telemetry Node and dynamically attach the 446 surviving sensors
    CREATE (t:Telemetry)
    SET t += $sensor_dict
    
    // 5. Link the Wafer to its Telemetry
    CREATE (w)-[:GENERATED_TELEMETRY]->(t)
    """
    tx.run(query, wafer_id=wafer_id, timestamp=timestamp, actual_result=actual_result, risk_score=risk_score, sensor_dict=sensor_dict)

# ==========================================
# 4. INJECT THE DATA
# ==========================================
if neo4j_enabled:
    with driver.session() as session:
        for index, row in sample_data.iterrows():
            
            wafer_id = f"WAFER_{index}"
            timestamp = str(row['Timestamp'])
            actual_result = int(row['Result'])
            risk_score = float(row['ML_Risk_Score'])
            
            # Isolate just the sensor readings for the Telemetry node using model-aligned names
            sensors_only = inference_features.loc[index].to_dict()
            
            session.execute_write(build_digital_twin, wafer_id, timestamp, actual_result, risk_score, sensors_only)

    print("Digital Twin Ontology successfully populated!")
    driver.close()
else:
    display(sample_data[['Timestamp', 'Result', 'ML_Risk_Score']])

C:\Users\zraym\AppData\Local\Temp\ipykernel_7400\2756209076.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sample_data = sample_data.assign(ML_Risk_Score=risk_scores)


Intelligence layer applied. Booting up Neo4j connection...
Neo4j connection verified.
Digital Twin Ontology successfully populated!
